# Training CNN/SNN
The purpose of this repository is to train CNNs and SNNs for ECE662.


## Setup
There are a few things you should understand. Firstly, we are using Google Drive as backing storage. Therefore, you need it shared with you. If it is shared with you, you need to add a shortcut under your `MyDrive` folder. Kindly then change the base path to point to it. Here are some presets (feel free to add yours!):
- *Homer:* `/content/drive/MyDrive/FPGA_Project/Training_Data/`

In [ ]:
!pip install snntorch

In [ ]:
import numpy as np
from google.colab import drive

# Configure your base path, this will determine how data is loaded.
BASE_PATH: str = '/content/drive/MyDrive/FPGA_Project/Training_Data/'

# 1. Mount Google Drive (this will prompt you to log in with your Google account)
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


I really love pretty `matplotlib` plots, so we are going to configure it a bit up here.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import matplotlib as mpl


font_path = f"{BASE_PATH}HelveticaNeueLight.otf"  # Change to actual filename

font_manager.fontManager.addfont(font_path)

font_name = font_manager.FontProperties(
    fname=font_path
).get_name()

print("Loaded font:", font_name)


mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica Neue", "Helvetica", "Arial"],
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 100,
})

Loaded font: Helvetica Neue


In [ ]:

import csv
import os
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from torch.utils.data import Dataset, get_worker_info


def build_rat_map(base_path):
    with open(Path(base_path) / "training_targets.csv", newline="") as f:
        return {
            int(row["rat_index"]): row
            for row in csv.DictReader(f)
        }


class RatDataset(Dataset):
    def __init__(
        self,
        identifiers: list[int],
        base_path: str,
        y_label: str,
        rat_map: dict | None = None,
        local_path: str = "/content/rat_data",
        downsampling: tuple[int, int] = (2, 8),
    ):
        self.identifiers = list(identifiers)
        self.base_path = Path(base_path)
        self.local_path = Path(local_path)
        self.local_path.mkdir(parents=True, exist_ok=True)

        self.downsampling = tuple(downsampling)
        if (
            len(self.downsampling) != 2
            or any(v < 1 for v in self.downsampling)
        ):
            raise ValueError(
                "downsampling must be two positive integers"
            )

        self.rat_map = (
            rat_map if rat_map is not None
            else build_rat_map(base_path)
        )

        self.files = {}
        self.targets = {}
        self.index = []

        self._handles = {}
        self._owner = None

        ds_c, ds_s = self.downsampling

        for rat_id in tqdm(
            self.identifiers,
            desc="Preparing rats",
            unit="rat"
        ):
            source = (
                self.base_path /
                f"Rat{rat_id}_FSA_frames.h5"
            )

            # Different downsampling configurations
            # get different cache files.
            destination = (
                self.local_path /
                f"Rat{rat_id}_norm_avgpool_"
                f"{ds_c}x{ds_s}_f32.h5"
            )

            if not destination.exists():
                if not source.exists():
                    raise FileNotFoundError(source)

                self._convert_h5(
                    source,
                    destination,
                    rat_id,
                    self.downsampling
                )

            with h5py.File(destination, "r") as f:
                rf = f["rf_data"]

                if rf.ndim != 4:
                    raise ValueError(
                        f"Expected 4D data for Rat {rat_id}"
                    )

                n_frames = rf.shape[0]

            self.files[rat_id] = destination

            self.targets[rat_id] = (
                float(self.rat_map[rat_id][y_label])
                / 100.0
            )

            for frame_idx in range(n_frames):
                self.index.append((rat_id, frame_idx))

        print(
            f"Ready: {len(self.index)} frames "
            f"from {len(self.identifiers)} rats"
        )
        print(
            f"Cached downsampling: {self.downsampling}"
        )

    @staticmethod
    def _convert_h5(
        source,
        destination,
        rat_id,
        downsampling
    ):
        temp_path = destination.with_suffix(".h5.part")
        ds_c, ds_s = downsampling

        try:
            with h5py.File(source, "r") as src, \
                 h5py.File(temp_path, "w") as dst:

                rf = src["rf_data"]

                if rf.ndim != 4:
                    raise ValueError("Expected 4D RF data")

                frames, events, channels, samples = rf.shape

                if channels % ds_c != 0:
                    raise ValueError(
                        "Receiver channels must divide "
                        "evenly by downsampling[0]"
                    )

                if samples % ds_s != 0:
                    raise ValueError(
                        "RF samples must divide evenly "
                        "by downsampling[1]"
                    )

                out_channels = channels // ds_c
                out_samples = samples // ds_s

                chunk_events = min(8, events)

                output = dst.create_dataset(
                    "rf_data",
                    shape=(
                        frames,
                        events,
                        out_channels,
                        out_samples
                    ),
                    dtype=np.float32,
                    chunks=(
                        1,
                        chunk_events,
                        out_channels,
                        out_samples
                    ),
                    compression=None
                )

                for key, value in rf.attrs.items():
                    output.attrs[key] = value

                output.attrs["downsampling"] = downsampling
                output.attrs["normalized"] = True
                output.attrs["preprocessing"] = (
                    "full-frame max-abs normalization, "
                    "then average pooling"
                )

                # Process one frame at a time and
                # a few events at a time to avoid
                # loading full frames into RAM.
                for frame_idx in tqdm(
                    range(frames),
                    desc=f"Caching Rat {rat_id}",
                    unit="frame",
                    leave=False
                ):
                    # Pass 1: obtain the full-frame
                    # maximum absolute value.
                    frame_max = 0.0

                    for start in range(
                        0, events, chunk_events
                    ):
                        end = min(
                            start + chunk_events,
                            events
                        )

                        raw = rf[frame_idx, start:end]

                        # int32 avoids int16 overflow
                        # for values like -32768.
                        chunk_max = np.max(
                            np.abs(
                                raw.astype(np.int32)
                            )
                        )

                        frame_max = max(
                            frame_max,
                            float(chunk_max)
                        )

                    frame_max = max(frame_max, 1e-6)

                    # Pass 2: normalize, downsample,
                    # and write the cached output.
                    for start in range(
                        0, events, chunk_events
                    ):
                        end = min(
                            start + chunk_events,
                            events
                        )

                        raw = rf[frame_idx, start:end]

                        x = torch.from_numpy(
                            np.asarray(
                                raw,
                                dtype=np.float32
                            )
                        )

                        x = x / frame_max

                        # (E, C, S) -> (E, 1, C, S)
                        x = x.unsqueeze(1)

                        x = F.avg_pool2d(
                            x,
                            kernel_size=downsampling,
                            stride=downsampling
                        )

                        # (E, 1, C_small, S_small)
                        # -> (E, C_small, S_small)
                        output[frame_idx, start:end] = (
                            x.squeeze(1).numpy()
                        )

            os.replace(temp_path, destination)

        except Exception:
            temp_path.unlink(missing_ok=True)
            raise

    def _get_handle(self, rat_id):
        worker = get_worker_info()
        owner = (
            os.getpid(),
            worker.id if worker is not None else None
        )

        if self._owner != owner:
            self.close()
            self._owner = owner

        if rat_id not in self._handles:
            self._handles[rat_id] = h5py.File(
                self.files[rat_id],
                "r",
                rdcc_nbytes=1 * 1024 * 1024
            )

        return self._handles[rat_id]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        rat_id, frame_idx = self.index[idx]

        f = self._get_handle(rat_id)

        # Already normalized and downsampled.
        frame = f["rf_data"][frame_idx]

        x = torch.from_numpy(frame)

        y = torch.tensor(
            self.targets[rat_id],
            dtype=torch.float32
        )

        return x, y

    def close(self):
        for f in self._handles.values():
            try:
                f.close()
            except Exception:
                pass

        self._handles = {}
        self._owner = None

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_handles"] = {}
        state["_owner"] = None
        return state


In [ ]:
import gc

rat_map = build_rat_map(BASE_PATH)

train_ids = [7, 14, 2, 19, 5, 11, 1, 16,
             9, 20, 4, 13, 8, 17, 3, 12]

val_ids = [6, 10, 15, 18]

train_dataset = RatDataset(
    base_path=BASE_PATH,
    identifiers=train_ids,
    y_label='liver_fat_percent',
    rat_map=rat_map
)
val_dataset = RatDataset(
    base_path=BASE_PATH,
    identifiers=val_ids,
    y_label='liver_fat_percent',
    rat_map=rat_map
)

import time

# Time reading one item directly
start = time.perf_counter()
x, y = train_dataset[0]
print(f"One frame: {time.perf_counter() - start:.2f}s")


from torch.utils.data import DataLoader

sampler = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=False,
)

x_sample, y_sample = next(iter(sampler))
del sampler
gc.collect()

# print(next(iter(train_loader)))

Preparing rats:   0%|          | 0/16 [00:00<?, ?rat/s]

Caching Rat 7:   0%|          | 0/10 [00:00<?, ?frame/s]

Caching Rat 14:   0%|          | 0/10 [00:00<?, ?frame/s]

Caching Rat 2:   0%|          | 0/10 [00:00<?, ?frame/s]

Caching Rat 19:   0%|          | 0/10 [00:00<?, ?frame/s]

Caching Rat 5:   0%|          | 0/10 [00:00<?, ?frame/s]

**Nature of the Data**

We will end up with an array of $(N, E, C, F)$ where:
- $N$ is the number of frames (which are like recordings)

In [ ]:
# Check the shape and data type of the raw RF data

def print_data_stats(t: torch.Tensor, name: str):
  print(f'Data Statistics [{name}]:')
  print(f'  Shape: {t.shape}')
  print(f'  Dtype: {t.dtype}')
  print(f'  Min: {t.min()}')
  print(f'  Max: {t.max()}')


print_data_stats(x_sample, 'X')
print_data_stats(y_sample, 'Y')

## Defining the Model
The next thing we do is define our CNN model. For this, we will go with a very simple model design.

In [ ]:

import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate


class RatClassifier(nn.Module):
    def __init__(
        self,
        in_channels=384,
        spiking=False,
        steps=4,
        threshold=1.0
    ):
        super().__init__()

        self.spiking = spiking
        self.steps = steps

        # Pre-downsampled, normalized RF frames
        self.downsample = nn.Identity()

        self.stem = nn.Conv2d(
            in_channels, 16, kernel_size=1
        )

        self.conv1 = nn.Conv2d(
            16, 32, 3, stride=2, padding=1
        )
        self.conv2 = nn.Conv2d(
            32, 64, 3, stride=2, padding=1
        )
        self.conv3 = nn.Conv2d(
            64, 64, 3, stride=2, padding=1
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(64, 1)

        # Spike recording disabled during training
        self.record_spikes = False
        self.last_spike_counts = None
        self.last_spike_elements = None

        if spiking:
            spike_grad = surrogate.fast_sigmoid()

            self.act1 = snn.Leaky(
                beta=0.9, threshold=threshold, spike_grad=spike_grad
            )
            self.act2 = snn.Leaky(
                beta=0.9, threshold=threshold, spike_grad=spike_grad
            )
            self.act3 = snn.Leaky(
                beta=0.9, threshold=threshold, spike_grad=spike_grad
            )
        else:
            self.act1 = nn.ReLU()
            self.act2 = nn.ReLU()
            self.act3 = nn.ReLU()

    def forward(self, x):
        z = self.stem(self.downsample(x))

        if not self.spiking:
            z = self.act1(self.conv1(z))
            z = self.act2(self.conv2(z))
            z = self.act3(self.conv3(z))

            return self.head(
                self.pool(z).flatten(1)
            )

        # Membrane states reset every batch
        mem1 = self.act1.init_leaky()
        mem2 = self.act2.init_leaky()
        mem3 = self.act3.init_leaky()

        # First convolution uses static input
        current1 = self.conv1(z)

        logits_sum = 0

        if self.record_spikes:
            counts = [
                torch.zeros((), device=x.device)
                for _ in range(3)
            ]
            elements = [0, 0, 0]

        for t in range(self.steps):
            spk1, mem1 = self.act1(
                current1, mem1
            )

            spk2, mem2 = self.act2(
                self.conv2(spk1), mem2
            )

            spk3, mem3 = self.act3(
                self.conv3(spk2), mem3
            )

            if self.record_spikes:
                for i, spk in enumerate(
                    (spk1, spk2, spk3)
                ):
                    counts[i] = (
                        counts[i] + spk.detach().sum()
                    )
                    elements[i] += spk.numel()

            logits_sum = logits_sum + self.head(
                self.pool(spk3).flatten(1)
            )

        if self.record_spikes:
            self.last_spike_counts = torch.stack(
                counts
            )
            self.last_spike_elements = elements

        return logits_sum / self.steps


# Training
Now that we have loaded and tested the CNN, we will try to train with it.

In [ ]:
!pip install pydantic

In [ ]:
from dataclasses import dataclass
import torch.nn as nn
from pydantic import BaseModel


def count_params(model):
    return sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )



class TrainOutput(BaseModel):
  init_params: dict
  random_seed: int
  epochs: int
  notes: str
  learning_rate: float
  train_batch_size: int
  val_batch_size: int
  weight_decay: float
  parameter_count: int
  train_history: list[float]
  val_history: list[float]
  start_time: str
  end_time: str


def apply_preprocessing(x):
    x = x.float()

    # scale = x.abs().amax(
    #     dim=(1, 2, 3),
    #     keepdim=True
    # ).clamp_min(1e-6)

    # x.div_(scale)
    return x

def validate_model(model, criterion, val_loader):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for x, y in tqdm(
            val_loader,
            leave=False,
            desc="Running evaluation..."
        ):
            x = x.to(device, dtype=torch.float32)
            y = y.to(device, dtype=torch.float32).reshape(-1, 1)

            prediction = model(x)
            loss = criterion(prediction, y)

            batch_size = x.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

    return total_loss / total_samples

def train_model(
    output: TrainOutput,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion,
    scaler,
    epochs

) -> TrainOutput:

  for epoch in range(epochs):
      model.train()
      train_loss = 0.0

      for x, y in tqdm(train_loader, leave=False, desc='Inner'):
          # x: (1, 8, C, 128, 2496)
          # Remove rat batch dimension
          x = apply_preprocessing(x.to(device))
          y = y.to(device, dtype=torch.float32).unsqueeze(1)

          optimizer.zero_grad(set_to_none=True)

          # with torch.autocast(
          #     device_type="cuda",
          #     dtype=torch.float16,
          #     enabled=device.type == "cuda"
          # ):
              # print(x.shape, x.dtype)
              # print(x.shape)
          prediction = model(x.float())
              # print(prediction)
          loss = criterion(prediction.float(), y)
          # print(loss)
              # print('loss dtype', loss.dtype)

          # scaler.scale(loss).backward()

          # scaler.unscale_(optimizer)
          loss.backward()

          torch.nn.utils.clip_grad_norm_(
              model.parameters(),
              max_norm=1.0
          )
          optimizer.step()
          # scaler.step(optimizer)
          # scaler.update()

          train_loss += loss.item() * x.size(0)

      # Validation
      val_loss = validate_model(model, criterion, val_loader)

      output.train_history.append(train_loss / len(train_loader))
      output.val_history.append(val_loss)

      print(
          f"Epoch {epoch+1}/{epochs} | "
          f"Train MSE: {train_loss / len(train_loader):.4f} | "
          f"Val MSE: {val_loss:.4f}"
      )
  return output

import random
from datetime import datetime,timezone


def train_model_full(
    init_state: dict,
    device,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int,
    lr: float = 1e-4,
    weight_decay: float = 1e-3,
    out_folder: str = f'{BASE_PATH}results/',
    extra_notes: str = ''
):
# torch.manual_seed(42)
# model = RatClassifier(
#     in_channels=x_sample.shape[1],
#     spiking=False
# ).cuda()
  seed = random.randint(0, 1000000000)
  torch.manual_seed(seed)
  model = RatClassifier(**init_state).to(device)


  output = TrainOutput(
      init_params=init_state,
      random_seed=seed,
      parameter_count=count_params(model),
      notes=extra_notes,
      train_history=[],
      val_history=[],
      start_time=str(datetime.now(timezone.utc)),
      end_time='',
      train_batch_size=train_loader.batch_size,
      val_batch_size=val_loader.batch_size,
      epochs=epochs,
      learning_rate=lr,
      weight_decay=weight_decay
  )
  optimizer = torch.optim.AdamW(
      model.parameters(),
      lr=lr,
      weight_decay=weight_decay
  )

  criterion = nn.MSELoss()

  o = train_model(
      output=output,
      model=model,
      train_loader=train_loader,
      val_loader=val_loader,
      optimizer=optimizer,
      criterion=criterion,
      scaler=None,
      epochs=epochs
  )
  o.end_time = str(datetime.now(timezone.utc))

  import uuid
  with open(f'{out_folder}{str(uuid.uuid4())}.json', 'w') as fo:
    fo.write(o.model_dump_json(indent=4))

  return o


In [ ]:
import torch.nn as nn

model = RatClassifier(
    in_channels=x_sample.shape[1],
    spiking=False
).cuda()

sample = x_sample
print(f'Sample Shape: {sample.shape}')
print(f'Output Shape: {model(apply_preprocessing(x_sample).cuda()).shape}')
del sample
del model
gc.collect()

## Training
Now that we have put together our datasets, we will start actually training the model. We will do training using PyTorch's mixed precision training mode.

Below we will configure our `train_loader` (which houses our training data) and our `val_loader` (which houses our validation data).

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=8
)
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    pin_memory=True
)

In [ ]:
import torch
gc.collect() # Doing a GC run here so that any scraps from previous runs are cleaned.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXTRA_NOTES = 'Using 10/30 samples per rat,  with rat IDs for train_ids = [7, 14, 2, 19, 5, 11, 1, 16, 9, 20, 4, 13, 8, 17, 3, 12], val_ids = [6, 10, 15, 18], simple architecture. same architecture for SNN and CNN. v2'


# for _ in range(1):
#   output = train_model_full(
#       init_state={
#           "in_channels": x_sample.shape[1],
#           "spiking": False
#       },
#       device=device,
#       train_loader=train_loader,
#       val_loader=val_loader,
#       epochs=25,
#       extra_notes=EXTRA_NOTES,
#       # lr=0.001
#   )
# print(output)

In [ ]:
!pip install -q snntorch

In [ ]:
# for lr in [0.0001, 0.001, 0.01, 0.1]:
#   for _ in range(3):
#     output_snn = train_model_full(
#         init_state={
#             "in_channels": x_sample.shape[1],
#             "spiking": True
#         },
#         device=device,
#         train_loader=train_loader,
#         val_loader=val_loader,
#         epochs=75,
#         extra_notes=EXTRA_NOTES,
#         lr=lr
#     )

In [ ]:
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# # Left subplot
# axes[0].plot(output.train_history, label="CNN")
# axes[0].plot(output_snn.train_history, label="SNN")
# axes[0].set_title("Training Loss")
# axes[0].set_xlabel("Epoch")
# axes[0].set_ylabel("MSE")

# # Right subplot
# axes[1].plot(output.val_history, label='CNN')
# axes[1].plot(output_snn.val_history, label='SNN')
# axes[1].set_title("Validation Loss")
# axes[1].set_xlabel("Epoch")
# axes[1].set_ylabel("MSE")

# plt.tight_layout()
# plt.legend()
# plt.show()


## Performance Benchmarking



In [ ]:
from typing import Optional
import time
from dataclasses import dataclass
import uuid
from pydantic import BaseModel
from datetime import datetime


class BenchmarkResult(BaseModel):
  device: str
  device_name: str
  trial_time: str
  init_params: dict
  notes: str
  sample_shape: list[int]
  run_timings_ms: list[float]

device = 'cuda'


def run_benchmark(
    init_params: dict,
    sample: torch.Tensor,
    device_name: str,
    device: str,
    notes: str,
    trials: int = 1000,
    output_path: Optional[str] = None
):
  trial_time = str(datetime.now())
  output = BenchmarkResult(
      device_name=device_name,
      device=device,
      trial_time=trial_time,
      init_params=init_params,
      notes=notes,
      sample_shape=list(sample.shape),
      run_timings_ms=[]
  )
  model = RatClassifier(**init_params) \
    .to(device=device)


  print('cuda?', next(model.parameters()).device)



  for _ in tqdm(range(trials), leave=False):

    sample = torch.randn(x_sample.size()).to(device)
    epoch_start = time.perf_counter()

    o = model(sample)

    epoch_time = time.perf_counter() - epoch_start
    output.run_timings_ms.append(epoch_time)

  if output_path is not None:
    id = str(uuid.uuid4())
    with open(f'{output_path}{id}.json', 'w') as fo:
      fo.write(output.model_dump_json(indent=4))
  return output






# # print(x_sample.shape)
# for op in [False, True]:
#   run_benchmark(
#       init_params={
#           "in_channels": x_sample.shape[1],
#           "spiking": op
#       },
#       device='cuda',
#       sample=x_sample,
#       notes='v2',
#       trials=250,
#       device_name='T4 GPU',
#       output_path=f'{BASE_PATH}results/benchmarking/'
#   )




In [ ]:
def quick_save_modal(
    model: BaseModel,
    path: Optional[str] = None
):
  id = str(uuid.uuid4())
  if path is not None:
    with open(f'{path}{id}.json', 'w') as fo:
      fo.write(model.model_dump_json(indent=4))

In [ ]:
!pip install -q ultralytics-thop

In [ ]:
import torch
from thop import profile

class ModelComplexityResults(BaseModel):
  init_params: dict
  device_name: str
  parameter_count: int
  device: str
  macs: float
  note: str

def benchmark_complexity(model, input_shape=(1, 384, 64, 312)):
    device = next(model.parameters()).device
    x = torch.zeros(input_shape, device=device)

    model.eval()

    with torch.no_grad():
        macs, params = profile(
            model,
            inputs=(x,),
            verbose=False
        )

    # print(f"Parameters: {params:,}")
    # print(f"MACs/frame: {macs / 1e9:.4f} GMACs")
    # print(f"Approx. FLOPs/frame: {2 * macs / 1e9:.4f} GFLOPs")

    return macs, params

def run_complexity_bench(
    init_params: dict,
    device: str,
    device_name: str,
    note: str,
    trials: int,
    path: str
):
  output = ModelComplexityResults(
      init_params=init_params,
      device=device,
      device_name=device_name,
      parameter_count=0,
      note=note,
      macs=0.0
  )
  # for _ in tqdm(range(trials)):
  model = RatClassifier(**init_params) \
    .to(device=device)


  cnn_macs, cnn_params = benchmark_complexity(model)
  output.parameter_count = int(cnn_params)
  # output.macs.append(cnn_macs)

  output.macs = cnn_macs


  quick_save_modal(output, path)
  return output

for spiking in [False, True]:
  o = run_complexity_bench(
      init_params={
          "spiking": spiking
      },
      device='cpu',
      device_name='CPU',
      note='v2',
      trials=250,
      path=None
      # path=f'{BASE_PATH}results/benchmarking_macs/'
  )
  print(o)

# cnn = RatClassifier(spiking=False).to("cpu")
# snn_model = RatClassifier(spiking=True, steps=4).to("cpu")

# cnn_macs, cnn_params = benchmark_complexity(cnn)
# snn_macs, snn_params = benchmark_complexity(snn_model)
# print(cnn_macs, cnn_params)
# print(snn_macs, snn_params)

In [ ]:

import gc
import json
import random
import time
import uuid

from datetime import datetime, timezone
from pathlib import Path

import torch
import torch.nn as nn
from pydantic import BaseModel, Field
from tqdm.auto import tqdm


class SNNBenchmarkResults(BaseModel):
    run_id: str
    init_params: dict
    random_seed: int

    device: str
    device_name: str
    note: str

    epochs: int
    learning_rate: float
    weight_decay: float
    train_batch_size: int
    val_batch_size: int
    parameter_count: int

    train_history: list[float] = Field(default_factory=list)
    val_history: list[float] = Field(default_factory=list)

    best_val_mse: float = 0.0
    final_val_mse: float = 0.0

    training_seconds: float = 0.0
    spike_eval_seconds: float = 0.0
    frames_evaluated: int = 0

    total_spikes: int = 0
    spikes_per_frame: float = 0.0
    spikes_per_second: float = 0.0

    lif1_spikes: int = 0
    lif2_spikes: int = 0
    lif3_spikes: int = 0

    lif1_rate_pct: float = 0.0
    lif2_rate_pct: float = 0.0
    lif3_rate_pct: float = 0.0

    start_time: str = ""
    end_time: str = ""


def sync_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


@torch.inference_mode()
def benchmark_spikes(model, loader):
    device = next(model.parameters()).device
    model.eval()

    total_counts = torch.zeros(3, device=device)
    total_elements = [0, 0, 0]
    total_frames = 0

    model.record_spikes = True

    try:
        sync_device(device)
        start = time.perf_counter()

        for x, _ in loader:
            x = x.to(device, dtype=torch.float32)

            model(x)

            total_counts += model.last_spike_counts

            for i in range(3):
                total_elements[i] += model.last_spike_elements[i]

            total_frames += x.size(0)

        sync_device(device)
        elapsed = time.perf_counter() - start

    finally:
        model.record_spikes = False

    counts = [int(v) for v in total_counts.cpu().tolist()]
    spike_total = sum(counts)

    rates = [
        100.0 * counts[i] / total_elements[i]
        if total_elements[i] else 0.0
        for i in range(3)
    ]

    return {
        "spike_eval_seconds": elapsed,
        "frames_evaluated": total_frames,
        "total_spikes": spike_total,
        "spikes_per_frame": spike_total / total_frames,
        "spikes_per_second": spike_total / elapsed,
        "lif1_spikes": counts[0],
        "lif2_spikes": counts[1],
        "lif3_spikes": counts[2],
        "lif1_rate_pct": rates[0],
        "lif2_rate_pct": rates[1],
        "lif3_rate_pct": rates[2],
    }


def run_snn_bench(
    init_params: dict,
    device: str,
    device_name: str,
    note: str,
    epochs: int,
    train_loader,
    val_loader,
    path: str,
    lr: float = 1e-4,
    weight_decay: float = 1e-3,
    seed: int = 42,
):
    device = torch.device(device)

    if init_params.get("spiking") is not True:
        raise ValueError("init_params must contain spiking=True")

    random.seed(seed)
    torch.manual_seed(seed)

    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    model = RatClassifier(**init_params).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    criterion = nn.MSELoss()

    output = SNNBenchmarkResults(
        run_id=str(uuid.uuid4()),
        init_params=init_params.copy(),
        random_seed=seed,
        device=str(device),
        device_name=device_name,
        note=note,
        epochs=epochs,
        learning_rate=lr,
        weight_decay=weight_decay,
        train_batch_size=train_loader.batch_size,
        val_batch_size=val_loader.batch_size,
        parameter_count=sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        ),
        start_time=datetime.now(timezone.utc).isoformat()
    )

    sync_device(device)
    train_start = time.perf_counter()

    for epoch in range(epochs):
        model.train()

        train_loss_sum = 0.0
        train_samples = 0

        for x, y in tqdm(
            train_loader,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=False
        ):
            # RF data already normalized and downsampled
            x = x.to(device, dtype=torch.float32)
            y = y.to(device, dtype=torch.float32).reshape(-1, 1)

            optimizer.zero_grad(set_to_none=True)

            prediction = model(x)
            loss = criterion(prediction, y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )
            optimizer.step()

            n = x.size(0)
            train_loss_sum += loss.item() * n
            train_samples += n

        # Validation
        model.eval()
        val_loss_sum = 0.0
        val_samples = 0

        with torch.inference_mode():
            for x, y in val_loader:
                x = x.to(device, dtype=torch.float32)
                y = y.to(device, dtype=torch.float32).reshape(-1, 1)

                prediction = model(x)
                loss = criterion(prediction, y)

                n = x.size(0)
                val_loss_sum += loss.item() * n
                val_samples += n

        train_mse = train_loss_sum / train_samples
        val_mse = val_loss_sum / val_samples

        output.train_history.append(train_mse)
        output.val_history.append(val_mse)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train MSE: {train_mse:.6f} | "
            f"Val MSE: {val_mse:.6f}"
        )

    sync_device(device)
    output.training_seconds = time.perf_counter() - train_start

    # Evaluate spikes from the trained model
    spike_stats = benchmark_spikes(model, val_loader)

    for key, value in spike_stats.items():
        setattr(output, key, value)

    output.best_val_mse = min(output.val_history)
    output.final_val_mse = output.val_history[-1]
    output.end_time = datetime.now(timezone.utc).isoformat()

    # Save one Pydantic JSON record per run
    output_folder = Path(path)
    output_folder.mkdir(parents=True, exist_ok=True)

    save_path = output_folder / f"{output.run_id}.json"

    with open(save_path, "w") as f:
        f.write(output.model_dump_json(indent=4))

    print(f"\nSaved: {save_path}")
    print(f"Total spikes: {output.total_spikes:,}")
    print(f"Spikes/frame: {output.spikes_per_frame:,.0f}")
    print(f"Spike events/sec: {output.spikes_per_second:,.0f}")

    del model, optimizer
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return output


In [ ]:
# SNN_CONFIGS = [
#     {"spiking": True, "steps": 1, "threshold": 0.1},
#     {"spiking": True, "steps": 2, "threshold": 0.1},
#     {"spiking": True, "steps": 4, "threshold": 0.1},
#     {"spiking": True, "steps": 6, "threshold": 0.1},
#     {"spiking": True, "steps": 8, "threshold": 0.1},
# ]

# results = []

# for _ in range(5):
#   for i, init_params in enumerate(SNN_CONFIGS):
#     result = run_snn_bench(
#         init_params=init_params,
#         device="cuda",
#         device_name="A100",
#         note=f"SNN steps experiment v1, run {i+1}",
#         epochs=10,
#         train_loader=train_loader,
#         val_loader=val_loader,
#         lr=1e-4,
#         weight_decay=1e-3,
#         seed=42,
#         path=f"{BASE_PATH}results/benchmarking_spikes/"
#     )

#     results.append(result)

In [ ]:
def model_memory(model):
    param_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )

    buffer_bytes = sum(
        b.numel() * b.element_size()
        for b in model.buffers()
    )

    total_mb = (param_bytes + buffer_bytes) / (1024 ** 2)

    print(f"Parameters: {param_bytes / 1024**2:.4f} MiB")
    print(f"Buffers: {buffer_bytes / 1024**2:.4f} MiB")
    print(f"Total model size: {total_mb:.4f} MiB")

    return total_mb

cnn = RatClassifier(spiking=False).to("cpu")
snn_model = RatClassifier(spiking=True, steps=4).to("cpu")
print("CNN")
cnn_mb = model_memory(cnn)
print("SNN")
snn_mb = model_memory(snn_model)